# Prompt Piper — Real Experiment: Qwen3.5-9B × GSM8K

A complete end-to-end experiment showing everything needed for a real compression run:

* **Task**: `gsm8k` — grade-school math, *generative* scoring (`generate_until`);
  part of the HuggingFace Open LLM Leaderboard H6 benchmark.
* **Model**: `Qwen/Qwen3.5-9B` with 4-bit NF4 quantisation
* **Fitting**: `DatasetVocabularyFitter` mines the training split for dataset-specific
  stopwords, abbreviations, section headings, example-field markers, chunk separators,
  and imperative-rewrite patterns — using the LLM as a backend where needed.
* **Search**: NSGA-II, pop=50, 100 generations, n\_subset=100
  (~5 050 individual evaluations × 100 samples each)
* **Comparison**: fitted factory vs default factory on the same config
* **Held-out eval**: Pareto solutions re-scored on a test slice not seen during search

**Sections**
1. Load model and datasets
2. Inspect a GSM8K sample
3. Fit vocabulary and patterns with the LLM
4. Build combined factory
5. Baseline accuracy
6. Run optimisation — fitted factory
7. Run optimisation — default factory (comparison)
8. Pareto front: fitted vs default
9. Apply best pipeline to a sample
10. Held-out test evaluation
11. Operator frequency analysis

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

pd.set_option('display.max_colwidth', 120)

In [ ]:
NUM_FS = 4
BS = 8

## 1. Load model and datasets

GSM8K has two splits:
- `train` (~7 500 samples) — used exclusively for fitting
- `test` (~1 319 samples) — split into a search pool (first 1 000) and a held-out
  evaluation slice (last 319)

The `num_fewshot=NUM_FS` keeps prompts short and fast to evaluate; bump to 8 for the
standard benchmark setting once you are happy with the pipeline.

**Chat/instruct models** require `apply_chat_template=True` so lm-eval wraps
each prompt in the model's conversation tokens before scoring.  Pass it as a
model kwarg to `LMEvalBackend.build`.

The evaluator's stop sequences (`until`) are read automatically from the task's
own lm-eval config — GSM8K ships with `["Question:", "</s>", "<|im_end|>"]`,
which covers Qwen and Mistral.  For a model with a different end-of-turn token,
replace `backend.evaluator` after construction (see the comment in the cell below).

In [ ]:
import torch
from transformers import BitsAndBytesConfig
from prompt_piper.backends.lm_eval import LMEvalBackend

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

backend = LMEvalBackend.build(
    model_type='hf',
    model_name='Qwen/Qwen3.5-9B',
    task_name='gsm8k_platinum',
    num_fewshot=NUM_FS,
    batch_size=BS,
    max_length=8192,
    max_gen_toks=2048,
    quantization_config=bnb_config,
    fewshot_as_multiturn=True,
    apply_chat_template=True,    # Qwen3.5-9B is a chat/instruct model
)

print(f'Task output type : {getattr(backend.task, "OUTPUT_TYPE", "unknown")}')
print(f'Tokenizer type   : {type(backend.tokenizer._counter).__name__}')

# generation_kwargs are read from the task's lm-eval config automatically.
# GSM8K's until=["Question:", "</s>", "<|im_end|>"] covers Qwen's EOT token,
# so no extra_stop_sequences are needed here.
#
# For a model with a different EOT token (e.g. Llama-3), do:
#
# from prompt_piper.backends.lm_eval import LMEvalEvaluator
# backend.evaluator = LMEvalEvaluator(
#     backend.model, backend.task,
#     extra_stop_sequences=["<|eot_id|>"],   # Llama-3 end-of-turn token
# )

In [ ]:
# Training split — for vocabulary fitting only, never used in search or evaluation.
train_data = backend.get_dataset(
    split='train',
    num_fewshot=NUM_FS,
    prompt_field='prompt',
).as_list()

# Test split — first 1 000 for the optimisation search, last ~319 held out.
test_data = backend.get_dataset(
    split='test',
    num_fewshot=NUM_FS,
    prompt_field='prompt',
).as_list()

search_data  = test_data[:1000]
heldout_data = test_data[1000:]

print(f'Train samples  : {len(train_data)}')
print(f'Search samples : {len(search_data)}')
print(f'Held-out samples: {len(heldout_data)}')

## 2. Inspect a GSM8K sample

GSM8K prompts are math word problems.  The target is the final numerical answer
after the `####` delimiter.  With `num_fewshot=NUM_FS` the prompt is just the question;
with `num_fewshot=NUM_FS` (standard) it includes eight worked examples, making
compression much more impactful.

In [ ]:
sample = train_data[0]
print('Keys:', list(sample.keys()))
print()
print('=== Prompt ===')
print(sample['prompt'])
print()
print(f'=== Target: {sample["target"]!r} ===')
print()
tok_count = backend.tokenizer.count(sample['prompt'])
print(f'Token count (Qwen3.5-9B tokenizer): {tok_count}')

# Show token-count distribution over a small sample
counts = [backend.tokenizer.count(s['prompt']) for s in train_data[:200]]
print(f'Token counts over first 200 train samples — '
      f'min={min(counts)}, median={int(np.median(counts))}, max={max(counts)}')

## 3. Fit vocabulary and patterns with the LLM

`DatasetVocabularyFitter` analyses the training corpus to produce
dataset-specific transformation instances:

| Operator | What is fitted | Backend needed? |
|---|---|---|
| O3 `RemoveStopwords` | High-frequency non-answer words | No |
| O5 `AbbreviationSubstitution` | Recurring multi-word phrases | No |
| O17 `ContextPruning` | Anchor terms from target vocab | No |
| O18 `SectionPruning` | Structural headings + XML tags | No + Yes |
| O21 `ExampleSimplification` | Input/output field names | No |
| O22 `LabelShortening` | Target label map | No |
| O4 `ReplaceWithShortSynonym` | Verbose → short phrase pairs | Yes |
| O10 `ImperativeRewrite` | Task-framing regex patterns | Yes |
| O15 `RemovePromptHeader` | Non-standard header formats | Yes |
| O28 `ChunkCompression` | Block separator pattern | Yes |

`sample_size=500` caps the number of training documents analysed — enough to
saturate frequency counts on GSM8K without processing all 7 500.

In [ ]:
from prompt_piper.transformations.fitting import DatasetVocabularyFitter

fitter = DatasetVocabularyFitter(
    train_data,
    prompt_field='prompt',
    target_field='target',
    sample_size=500,
    seed=42,
    backend=backend.semantic_backend,  # enables O4, O10, O15, O28
)

print('Fitting vocabulary and patterns on 500 training samples...')
report = fitter.report()
print('Done.')

In [ ]:
# ── Corpus-based discoveries ────────────────────────────────────────────────
print('=== New stopwords (beyond defaults) ===')
print(', '.join(sorted(report['stopwords'])) or '  (none)')

print()
print('=== New abbreviations ===')
for pat, repl in report['abbreviations']:
    print(f'  {pat!r:55s}  ->  {repl!r}')
if not report['abbreviations']:
    print('  (none beyond defaults)')

print()
print('=== New section headings ===')
print(', '.join(repr(s) for s in report['sections']) or '  (none)')

print()
print('=== Example field markers ===')
print(f'  input  : {report["example_fields"]["input_markers"]}')
print(f'  output : {report["example_fields"]["output_markers"]}')

print()
print('=== Context anchor terms ===')
print(report.get('context_anchors', []))

# ── LLM-backed discoveries ───────────────────────────────────────────────────
if 'short_synonyms' in report:
    print()
    print('=== Short synonyms (LLM-discovered) ===')
    for pat, repl in report['short_synonyms']:
        print(f'  {pat!r:55s}  ->  {repl!r}')
    if not report['short_synonyms']:
        print('  (none beyond defaults)')

if 'imperative_patterns' in report:
    print()
    print('=== Imperative-rewrite patterns (LLM-discovered) ===')
    for p in report['imperative_patterns']:
        print(f'  {p!r}')
    if not report['imperative_patterns']:
        print('  (none beyond defaults)')

if 'chunk_separator' in report:
    print()
    print(f'=== Chunk separator pattern (LLM-discovered) ===')
    print(f'  {report["chunk_separator"]!r}')

if 'tag_names' in report:
    print()
    print(f'=== XML/HTML tag names (LLM-discovered) ===')
    print(f'  {report["tag_names"]}')

## 4. Build combined factory

Two factories need to be combined:

1. **`DatasetVocabularyFitter`** — returns corpus/LLM-fitted instances for
   O3, O5, O10, O15, O17, O18, O21, O22, O28.
2. **LLM semantic backend** — injects the model into O23–O27 so they do real
   LLM-based compression at `apply()` time (rather than heuristic fallbacks).

The combined factory gives priority to the fitted instance when one exists,
then falls back to LLM injection for remaining semantic operators.

In [ ]:
# Pre-build the fitted instances once so the factory is a fast dict lookup.
fitted_instances = fitter.fit_all()
llm_semantic_backend = backend.semantic_backend

def combined_factory(cls):
    # 1. Corpus/LLM-fitted instance (O3, O5, O10, O15, O17, O18, O21, O22, O28)
    if cls.id in fitted_instances:
        return fitted_instances[cls.id]
    # 2. LLM backend injection for remaining semantic operators (O23-O27)
    if (cls.category == 'semantic'
            and 'backend' in inspect.signature(cls.__init__).parameters):
        return cls(backend=llm_semantic_backend)
    # 3. Default no-arg construction for everything else
    return cls()

print('Fitted operator IDs:', sorted(fitted_instances.keys()))

## 5. Baseline accuracy

Measure exact-match accuracy on uncompressed prompts.  This is the upper bound
that compression should try to preserve.

`evaluate_batch` sends all 50 prompts to the model in one batched call.

In [ ]:
baseline_samples = heldout_data[:50]
originals = [s['prompt'] for s in baseline_samples]

scores = backend.evaluator.evaluate_batch(originals, originals, baseline_samples)
baseline_acc = float(np.mean(scores))

print(f'Baseline exact-match accuracy (n=50, uncompressed): {baseline_acc:.3f}')
print('(Typical range for a 7-9B model on 0-shot GSM8K: 0.30 - 0.55)')

## 6. Run optimisation — fitted factory

Search budget with these settings:

| | Value | Evals |
|---|---|---|
| Initial population | pop\_size = 50 | 50 × 100 = 5 000 |
| 100 generations | 50 offspring/gen | 5 000 × 100 = 500 000 |
| **Total sample evals** | | **~505 000** |
| **Batched model calls** | | **5 050** |

Each model call carries `n_subset` prompts — lm-eval chunks them into
`batch_size` GPU micro-batches internally.

In [ ]:
from prompt_piper.optimization import ExperimentConfig

config = ExperimentConfig(
    algorithm='nsga2',
    pop_size=50,
    n_gen=100,
    n_subset=100,       # samples per fitness evaluation (drawn from search_data)
    subset_seed=42,
    seed=42,
)

total_evals = config.pop_size * (1 + config.n_gen)
print(f'Individual evaluations : {total_evals:,}')
print(f'Sample evaluations     : {total_evals * config.n_subset:,}')

In [ ]:
print('Running NSGA-II with fitted factory...')
experiment_fitted = backend.build_experiment(
    config,
    split='test',
    num_fewshot=NUM_FS,
    transformation_factory=combined_factory,
)
# Limit the dataset used by Experiment to the search_data slice.
# Experiment._full_dataset is set in __init__; re-build directly to pass a slice.
from prompt_piper.optimization.runner import Experiment

experiment_fitted = Experiment(
    config=config,
    dataset=search_data,
    evaluator=backend.evaluator,
    tokenizer=backend.tokenizer,
    transformation_factory=combined_factory,
)
result_fitted = experiment_fitted.run(
    baselines=['no_compression', 'all_transformations']
)
print(result_fitted.summary())

## 7. Run optimisation — default factory

Same config and seed; only the factory differs.  The default factory uses
generic operator configurations (built-in stopword list, default section
headings, etc.) without any dataset-specific tuning.

In [ ]:
print('Running NSGA-II with default factory...')
experiment_default = Experiment(
    config=config,
    dataset=search_data,
    evaluator=backend.evaluator,
    tokenizer=backend.tokenizer,
    # no transformation_factory -> uses cls() for every operator
)
result_default = experiment_default.run(
    baselines=['no_compression', 'all_transformations']
)
print(result_default.summary())

## 8. Pareto front: fitted vs default

In [ ]:
def pareto_df(result, label):
    rows = []
    for bits, names, tokens, perf in zip(
        result.pareto_solutions,
        result.pareto_transformation_names,
        result.avg_tokens,
        result.avg_performance,
    ):
        rows.append({
            'factory':    label,
            'avg_tokens': round(float(tokens), 1),
            'accuracy':   round(float(perf), 3),
            'n_ops':      int(bits.sum()),
            'operators':  ', '.join(names) if names else '(none)',
        })
    return pd.DataFrame(rows).sort_values('avg_tokens')

df_fitted  = pareto_df(result_fitted,  'fitted')
df_default = pareto_df(result_default, 'default')

print('=== Fitted factory Pareto front ===')
display(df_fitted[['avg_tokens', 'accuracy', 'n_ops', 'operators']])
print()
print('=== Default factory Pareto front ===')
display(df_default[['avg_tokens', 'accuracy', 'n_ops', 'operators']])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for res, label, color in [
    (result_fitted,  'Fitted factory (NSGA-II)',  'steelblue'),
    (result_default, 'Default factory (NSGA-II)', 'darkorange'),
]:
    idx = res.avg_tokens.argsort()
    ax.plot(res.avg_tokens[idx], res.avg_performance[idx],
            'o-', label=label, zorder=3)

nc = result_fitted.baselines['no_compression']
at = result_fitted.baselines['all_transformations']
ax.scatter(nc.avg_tokens.mean(), nc.avg_performance.mean(),
           marker='*', s=250, color='gray', zorder=5, label='no_compression')
ax.scatter(at.avg_tokens.mean(), at.avg_performance.mean(),
           marker='D', s=80, color='red',  zorder=5, label='all_transformations')
ax.axhline(baseline_acc, color='gray', linestyle='--',
           linewidth=0.8, label=f'baseline acc ({baseline_acc:.3f})')

ax.set_xlabel('Average tokens (compressed prompt)')
ax.set_ylabel('Exact-match accuracy (GSM8K)')
ax.set_title('Qwen3.5-9B on GSM8K — Fitted vs Default factory')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

nc_tok = float(nc.avg_tokens.mean())
_, tok_f, perf_f = result_fitted.best_compression()
_, tok_d, perf_d = result_default.best_compression()
print(f'No compression            : {nc_tok:.0f} tokens, acc={baseline_acc:.3f}')
print(f'Best compression (fitted) : {tok_f:.0f} tokens ({tok_f/nc_tok:.1%}), acc={perf_f:.3f}')
print(f'Best compression (default): {tok_d:.0f} tokens ({tok_d/nc_tok:.1%}), acc={perf_d:.3f}')

## 9. Apply best pipeline to a sample

Reconstruct the best-compression pipeline from the binary mask and apply it to
a test sample that was **not** part of the search subset.

In [ ]:
from prompt_piper.transformations import ALL_TRANSFORMATIONS

bits_comp, tok_comp, perf_comp = result_fitted.best_compression()
bits_perf, tok_perf, perf_perf = result_fitted.best_performance()

print(f'Best-compression solution : {tok_comp:.0f} avg tokens, acc={perf_comp:.3f}')
print(f'Best-performance solution : {tok_perf:.0f} avg tokens, acc={perf_perf:.3f}')

# Instantiate via the fitted factory so operators carry corpus-fitted patterns.
all_instances = [combined_factory(cls) for cls in ALL_TRANSFORMATIONS]
pipeline = [inst for inst, b in zip(all_instances, bits_comp) if b]

print(f'\nOperators in best-compression pipeline ({len(pipeline)}):')
for inst in pipeline:
    print(f'  {inst.name}')

In [ ]:
# Use a sample from the held-out slice (index >= 1000, outside the search pool).
sample = heldout_data[5]
original = sample['prompt']

compressed = original
for op in pipeline:
    compressed = op(compressed)

n_orig = backend.tokenizer.count(original)
n_comp = backend.tokenizer.count(compressed)

print(f'=== Original ({n_orig} tokens) ===')
print(original)
print()
print(f'=== Compressed ({n_comp} tokens, {n_comp/n_orig:.1%} of original) ===')
print(compressed)

In [ ]:
# Batch-score 20 held-out samples: original vs compressed.
eval_samples  = heldout_data[:20]
orig_texts    = [s['prompt'] for s in eval_samples]
comp_texts    = []
for t in orig_texts:
    c = t
    for op in pipeline:
        c = op(c)
    comp_texts.append(c)

orig_scores = backend.evaluator.evaluate_batch(orig_texts, orig_texts, eval_samples)
comp_scores = backend.evaluator.evaluate_batch(orig_texts, comp_texts, eval_samples)

print(f'Accuracy over 20 held-out samples')
print(f'  Original  : {np.mean(orig_scores):.3f}')
print(f'  Compressed: {np.mean(comp_scores):.3f}')

rows = []
for i, (ot, ct, os_, cs_) in enumerate(
        zip(orig_texts, comp_texts, orig_scores, comp_scores)):
    rows.append({
        'orig_tok': backend.tokenizer.count(ot),
        'comp_tok': backend.tokenizer.count(ct),
        'orig_acc': int(os_),
        'comp_acc': int(cs_),
    })
display(pd.DataFrame(rows))

## 10. Held-out test evaluation

The optimisation searched over `search_data` (first 1 000 test samples).
Here we re-score every Pareto solution on `heldout_data` (last 319 samples)
to check how well the solutions generalise.

A small positive gap (search > held-out) is normal; a large gap suggests
overfitting to the search subset — fix by increasing `n_subset`.

In [ ]:
print(f'Re-scoring {len(result_fitted.pareto_solutions)} Pareto solutions on '
      f'{len(heldout_data)} held-out samples...')

heldout_objectives = experiment_fitted.evaluate_solutions(
    result_fitted.pareto_solutions,
    heldout_data,
)

tokens_heldout = heldout_objectives[:, 0]
perf_heldout   = heldout_objectives[:, 1]
print('Done.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Search-phase Pareto front (dashed)
idx = result_fitted.avg_tokens.argsort()
ax.plot(result_fitted.avg_tokens[idx], result_fitted.avg_performance[idx],
        'o--', color='steelblue', alpha=0.5,
        label=f'Search phase (n_subset={config.n_subset})')

# Same solutions on held-out data (solid)
idx2 = tokens_heldout.argsort()
ax.plot(tokens_heldout[idx2], perf_heldout[idx2],
        'o-', color='steelblue',
        label=f'Held-out test (n={len(heldout_data)})')

ax.scatter(nc.avg_tokens.mean(), nc.avg_performance.mean(),
           marker='*', s=250, color='gray', zorder=5, label='no_compression')
ax.axhline(baseline_acc, color='gray', linestyle='--', linewidth=0.8,
           label=f'baseline acc ({baseline_acc:.3f})')

ax.set_xlabel('Average tokens')
ax.set_ylabel('Exact-match accuracy (GSM8K)')
ax.set_title('Search-phase vs held-out performance — fitted factory')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

gap = float(np.mean(result_fitted.avg_performance) - np.mean(perf_heldout))
print(f'Mean accuracy — search: {np.mean(result_fitted.avg_performance):.3f}  '
      f'held-out: {np.mean(perf_heldout):.3f}  gap: {gap:+.3f}')
print('A small positive gap is expected; a large one means increase n_subset.')

## 11. Operator frequency analysis

Which operators appear most often on the Pareto front?  Operators that recur
across many solutions are robustly useful on GSM8K regardless of the
token/accuracy trade-off point.  Compare fitted vs default to see whether
corpus fitting changed the operator selection.

In [ ]:
counts_fitted  = Counter()
counts_default = Counter()

for names in result_fitted.pareto_transformation_names:
    counts_fitted.update(names)
for names in result_default.pareto_transformation_names:
    counts_default.update(names)

all_ops = sorted(set(counts_fitted) | set(counts_default))

if all_ops:
    fig, axes = plt.subplots(
        1, 2,
        figsize=(14, max(4, len(all_ops) * 0.4 + 1)),
        sharey=True,
    )
    for ax, counts, label, color in [
        (axes[0], counts_fitted,  'Fitted factory',  'steelblue'),
        (axes[1], counts_default, 'Default factory', 'darkorange'),
    ]:
        if counts:
            ops, freqs = zip(*counts.most_common())
            ax.barh(list(ops)[::-1], list(freqs)[::-1], color=color)
        ax.set_xlabel('Frequency on Pareto front')
        ax.set_title(label)
    plt.suptitle('Operator frequency — Qwen3.5-9B / GSM8K', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No operators selected on the Pareto front.')

# Summary table: which operators appear in fitted but not default (or vice versa)
only_fitted  = set(counts_fitted)  - set(counts_default)
only_default = set(counts_default) - set(counts_fitted)
if only_fitted:
    print('Operators unique to fitted front  :', ', '.join(sorted(only_fitted)))
if only_default:
    print('Operators unique to default front :', ', '.join(sorted(only_default)))